# Phase 5.5 Kaggle Runner

This notebook runs one frozen model slot per Kaggle GPU session. It verifies the repository, hardware, dependencies, parser fixtures, provenance gates, and evidence boundary before any official dispatch. The current repository remains authorization-locked until its policy says official dispatch is enabled.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import secrets
import shutil
import subprocess
import sys
import tarfile

REPOSITORY_URL = os.environ.get(
    "PHASE5_REPOSITORY_URL",
    "https://github.com/rana-m-ahmed/ResearchWork-on-Mcp-Privilege-Aggregation.git",
)
MODEL_SLOT = "M2"
EXECUTE_OFFICIAL = False
DATASET_VERSION = "P5-DV-1.0.2-A7C91E42"
EXPECTED_SOURCE_COMMIT = "b90158e6"
EXPECTED_BRANCH_HEADS = {
    "M1": "79742bbb",
    "M2": "2ece7032",
    "M3": "45f0dcf1",
    "M4": "c5660c12",
}
BRANCHES = {slot: f"phase5_5-model-{slot.removeprefix('M')}" for slot in ("M1", "M2", "M3", "M4")}
MODEL_IDS = {
    "M1": "Qwen/Qwen2.5-7B-Instruct",
    "M2": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "M3": "mistralai/Mistral-7B-Instruct-v0.3",
    "M4": "microsoft/Phi-3.5-mini-instruct",
}
if MODEL_SLOT not in BRANCHES:
    raise ValueError(f"MODEL_SLOT must be one of {tuple(BRANCHES)}, got {MODEL_SLOT!r}")

REPO_ROOT = Path("/kaggle/working/ResearchWork-on-Mcp-Privilege-Aggregation")
OUTPUT_ROOT = Path("/kaggle/working/phase5_5_outputs")
UTC_DATE = datetime.now(timezone.utc).strftime("%Y%m%d")
RUN_TOKEN = secrets.token_hex(4).upper()
RUN_ID = f"P5PRE-{DATASET_VERSION}-{MODEL_SLOT}-{UTC_DATE}-{RUN_TOKEN}"
BRANCH = BRANCHES[MODEL_SLOT]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({"model_slot": MODEL_SLOT, "branch": BRANCH, "run_id": RUN_ID, "official_requested": EXECUTE_OFFICIAL}, indent=2))


In [ ]:
if REPO_ROOT.exists():
    raise RuntimeError(f"Refusing to reuse existing repository path: {REPO_ROOT}")
subprocess.run(["git", "clone", "--no-single-branch", REPOSITORY_URL, str(REPO_ROOT)], check=True)
for branch_name in BRANCHES.values():
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "origin", f"refs/heads/{branch_name}:refs/heads/{branch_name}"],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", BRANCH], check=True)


In [ ]:
def git(*arguments: str) -> str:
    return subprocess.check_output(["git", "-C", str(REPO_ROOT), *arguments], text=True).strip()

actual_branch_head = git("rev-parse", "HEAD")
expected_prefix = EXPECTED_BRANCH_HEADS[MODEL_SLOT]
if not actual_branch_head.startswith(expected_prefix):
    raise RuntimeError(f"branch head mismatch: expected {expected_prefix}, got {actual_branch_head}")
freeze = json.loads((REPO_ROOT / "phase5_5/manifests/phase5_5_source_freeze.json").read_text(encoding="utf-8-sig"))
if freeze["source_commit"] != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(f"source-freeze commit mismatch: {freeze['source_commit']}")
branch_config = json.loads((REPO_ROOT / "phase5_5/branch_config.json").read_text(encoding="utf-8-sig"))
if branch_config["model_slot"] != MODEL_SLOT or branch_config["exact_model_identifier"] != MODEL_IDS[MODEL_SLOT]:
    raise RuntimeError("selected branch does not match its approved model slot")
print(json.dumps({"branch_head": actual_branch_head, "source_commit": freeze["source_commit"], "model_id": MODEL_IDS[MODEL_SLOT]}, indent=2))


In [ ]:
def run_checked(command: list[str], *, cwd: Path | None = None) -> str:
    completed = subprocess.run(command, cwd=cwd, check=False, capture_output=True, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"command failed ({completed.returncode}): {' '.join(command)}\n{completed.stderr}")
    return completed.stdout

nvidia = run_checked(["nvidia-smi"])
print(nvidia)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(REPO_ROOT / "phase4_5/kaggle/requirements.lock.txt")], check=True)
hardware = run_checked([
    sys.executable,
    "-c",
    "import torch; assert torch.cuda.is_available(); print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'devices': torch.cuda.device_count()})",
])
print(hardware)


In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONDONTWRITEBYTECODE"] = "1"
pytest_command = [
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "phase5/tests",
    "phase5_5/tests",
    "-p",
    "no:cacheprovider",
    "--basetemp",
    str(OUTPUT_ROOT / "pytest-temp"),
]
subprocess.run(pytest_command, cwd=REPO_ROOT, env=test_environment, check=True)
gate_report = OUTPUT_ROOT / "gate0_authorization_report"
subprocess.run(
    [sys.executable, "-m", "phase5", "gate0", "--strict", "--root", str(REPO_ROOT), "--report-dir", str(gate_report)],
    cwd=REPO_ROOT,
    check=True,
)
canary_path = OUTPUT_ROOT / "qualification_canary.json"
subprocess.run(
    [
        sys.executable,
        "phase5_5/scripts/run_qualification_canary.py",
        "--root",
        str(REPO_ROOT),
        "--output",
        str(canary_path),
    ],
    cwd=REPO_ROOT,
    check=True,
)
canary = json.loads(canary_path.read_text(encoding="utf-8"))
if canary.get("pass") is not True or len(canary.get("records", [])) != 4:
    raise RuntimeError("live qualification canary failed")


In [ ]:
preflight_path = OUTPUT_ROOT / "official_preflight.json"
preflight_run = subprocess.run(
    [sys.executable, "phase5_5/scripts/official_preflight.py", "--root", str(REPO_ROOT), "--output", str(preflight_path)],
    cwd=REPO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
preflight = json.loads(preflight_path.read_text(encoding="utf-8"))
non_authorization_failures = [failure for failure in preflight["failures"] if failure != "official-dispatch-disabled"]
if non_authorization_failures:
    raise RuntimeError(f"official preflight failed for infrastructure/provenance reasons: {non_authorization_failures}")
if not preflight["checks"]["cuda_available"]:
    raise RuntimeError("Kaggle CUDA check failed")
if "official-dispatch-disabled" in preflight["failures"]:
    print("READY_FOR_AUTHORIZATION: repository policy still forbids official dispatch")
    if EXECUTE_OFFICIAL:
        raise RuntimeError("PHASE5_EXECUTE_OFFICIAL=1 but repository authorization is still disabled")
else:
    print("OFFICIAL_PREFLIGHT_PASS")


In [ ]:

from kaggle_secrets import UserSecretsClient
import selectors
import time

hf_token = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
if not hf_token:
    raise RuntimeError("HF_TOKEN is required for the real-backend pretrial")
os.environ["HF_TOKEN"] = hf_token
pretrial_attempts_root = Path("/kaggle/working/phase5_5_pretrial_attempts")
pretrial_evidence_root = Path("/kaggle/working/phase5_5_pretrial_evidence")
if pretrial_attempts_root.exists() or pretrial_evidence_root.exists():
    raise RuntimeError("pretrial output roots already exist; use a fresh Kaggle session")
pretrial_report = OUTPUT_ROOT / f"{MODEL_SLOT}_pretrial_report.json"
pretrial_command = [
    sys.executable,
    "-m",
    "phase5",
    "run-campaign",
    "--pretrial",
    "--model-slot",
    MODEL_SLOT,
    "--dataset-version",
    DATASET_VERSION,
    "--run-id",
    RUN_ID,
    "--seal-epoch",
    "1",
    "--until-safety-horizon",
    "--max-batches",
    "1",
    "--pretrial-trials",
    "3",
    "--attempts-root",
    str(pretrial_attempts_root),
    "--evidence-root",
    str(pretrial_evidence_root),
    "--batch-manifest",
    "phase5/manifests/batch_partition_manifest_v3.json",
    "--run-plan",
    "phase5/validation/kaggle_run_plan_v3.json",
    "--output",
    str(pretrial_report),
]
print(f"PRETRIAL_START: slot={MODEL_SLOT}; one_frozen_batch=50_rows", flush=True)
started = time.monotonic()
process = subprocess.Popen(
    pretrial_command,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
selector = selectors.DefaultSelector()
selector.register(process.stdout, selectors.EVENT_READ)
tail = []
while process.poll() is None:
    events = selector.select(timeout=30.0)
    if events:
        line = process.stdout.readline()
        if line:
            line = line.rstrip("
")
            print(line, flush=True)
            tail.append(line)
            del tail[:-100:]
    else:
        elapsed = int(time.monotonic() - started)
        print(f"PRETRIAL_HEARTBEAT: elapsed_seconds={elapsed}", flush=True)
for line in process.stdout:
    line = line.rstrip("
")
    print(line, flush=True)
    tail.append(line)
returncode = process.wait()
if returncode != 0:
    (OUTPUT_ROOT / f"{MODEL_SLOT}_pretrial_error.json").write_text(
        json.dumps({"returncode": returncode, "output_tail": tail}, indent=2, sort_keys=True) + "
",
        encoding="utf-8",
    )
    raise RuntimeError(f"real-backend pretrial failed with exit code {returncode}")
report = json.loads(pretrial_report.read_text(encoding="utf-8"))
if (
    report.get("model_slot") != MODEL_SLOT
    or len(report.get("processed_batch_ids", [])) != 1
    or report.get("resume_required") is not True
):
    raise RuntimeError("pretrial report does not reconcile to one bounded completed sample")
print(f"PRETRIAL_COMPLETE: {pretrial_report}", flush=True)


In [ ]:

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

evidence_root = Path("/kaggle/working/phase5_5_pretrial_evidence")
if not evidence_root.is_dir():
    raise RuntimeError("pretrial produced no evidence directory")
files = sorted(path for path in evidence_root.rglob("*") if path.is_file())
manifest = {
    "artifact": "phase5_5_real_backend_pretrial_v1",
    "model_slot": MODEL_SLOT,
    "run_id": RUN_ID,
    "dataset_version": DATASET_VERSION,
    "official_trial": False,
    "counts_for_phase5": False,
    "publication_evidence": False,
    "pretrial_batch_limit": 1,
    "pretrial_trial_limit": 3,
    "files": [{"path": str(path.relative_to(evidence_root)), "sha256": sha256(path)} for path in files],
}
manifest_path = OUTPUT_ROOT / f"{MODEL_SLOT}_pretrial_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "
", encoding="utf-8")
archive_path = OUTPUT_ROOT / f"{MODEL_SLOT}_pretrial_evidence.tar.gz"
with tarfile.open(archive_path, "w:gz") as archive:
    archive.add(evidence_root, arcname="phase5_5_pretrial_evidence")
    archive.add(manifest_path, arcname=f"phase5_5/{manifest_path.name}")
print(json.dumps({"manifest": str(manifest_path), "archive": str(archive_path), "archive_sha256": sha256(archive_path)}, indent=2))
